# Module 19: PCA and Clustering

**Case:** Can several dimensions of management practice be summarized, and do
firms form useful descriptive groups?

The management-index and PCA material adapts Gábor's World Management Survey
feature-engineering lecture. K-means is an extension. Both methods are
unsupervised: no outcome is used while fitting them.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

**Project data folder.** This setup finds the course folder whether the
notebook starts in the project root or inside a module. `data_dir` points
to the local files prepared by `scripts/fetch_data.py`.

In [ ]:
from pathlib import Path

project_root = Path.cwd()  # Start in the notebook's current working folder.
while not (project_root / "scripts" / "fetch_data.py").is_file():
    if project_root == project_root.parent:
        raise FileNotFoundError("Open this notebook inside the Coding 2 repository.")
    project_root = project_root.parent  # Look one folder higher.

data_dir = project_root / "data" / "raw"

In [ ]:
wms = pd.read_csv(data_dir / "wms_course.csv")
practices = ["operations", "monitor", "target", "people"]
wms[practices + ["management", "employees"]].describe().T.round(2)

## 1. Construct a transparent management index

In [ ]:
wms["management_index"] = wms[practices].mean(axis=1)
wms[["management_index", "management"]].corr().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(wms["management_index"], bins=25, ax=axes[0])
sns.scatterplot(data=wms, x="management_index", y="management", alpha=0.35, ax=axes[1])
axes[0].set(title="Constructed management index")
axes[1].set(title="Constructed and published measures")
plt.tight_layout()
plt.show()

## 2. Standardize before variance- and distance-based methods

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(wms[practices])
pd.DataFrame(X_scaled, columns=practices).agg(["mean", "std"]).round(2)

## 3. PCA constructs weighted combinations

In [ ]:
pca = PCA()
scores = pca.fit_transform(X_scaled)
explained = pd.DataFrame(
    {"component": range(1, 5), "variance_share": pca.explained_variance_ratio_, "cumulative": np.cumsum(pca.explained_variance_ratio_)}
)
explained.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=explained, x="component", y="variance_share", color="steelblue", ax=ax)
ax.set(title="Variance explained by each component", ylabel="Share of variance")
plt.show()

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=practices,
    columns=[f"PC{i}" for i in range(1, 5)],
)
loadings.round(3)

In [ ]:
wms[["pc1", "pc2"]] = scores[:, :2]
wms[["management_index", "pc1", "pc2"]].corr().round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
points = ax.scatter(wms["pc1"], wms["pc2"], c=np.log(wms["employees"]), cmap="viridis", alpha=0.55)
ax.set(title="Firms in principal-component space", xlabel="PC1 score", ylabel="PC2 score")
fig.colorbar(points, ax=ax, label="Log employees")
plt.show()

## 4. Use k-means as a descriptive grouping tool

In [ ]:
candidates = []
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X_scaled)
    candidates.append({"k": k, "silhouette": silhouette_score(X_scaled, labels)})
candidate_scores = pd.DataFrame(candidates)
candidate_scores.round(3)

In [ ]:
best_k = int(candidate_scores.loc[candidate_scores["silhouette"].idxmax(), "k"])
kmeans = KMeans(n_clusters=best_k, n_init=20, random_state=42)
wms["cluster"] = kmeans.fit_predict(X_scaled)
print(f"Selected k by silhouette score: {best_k}")

In [ ]:
profile = wms.groupby("cluster")[practices + ["management_index", "employees"]].agg(["mean", "count"])
profile.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=wms, x="pc1", y="pc2", hue="cluster", palette="tab10", alpha=0.6, ax=ax)
ax.set(title="K-means groups shown in PCA space")
plt.show()

Cluster numbers are arbitrary, and a short label is only a description of this
sample. Calling a cluster “high-performing” would require an outcome and a
separate analysis; k-means did not establish performance or causality.

## Session 5 reconstruction handoff

Open `prompts/session05-unsupervised-reconstruction.md`. Audit whether the LLM
standardizes before PCA and k-means, distinguishes loadings from scores, keeps
outcomes out of fitting, and avoids invented cluster narratives.

### Optional extension

Compare country-level means of PC1 and the transparent management index. Ask
whether their substantive conclusions agree and what sample-composition
differences could affect the comparison.